# 🌧️ 손으로 익히는 미니 트랜스포머 — "하늘에 먹구름이 많아지면 뭐가 생각나?"

> 📖 **라인별 상세 주석판(읽고 실행하는 버전)**: 이 노트북은 빈 칸을 채우는 대신,
> 각 코드셀에 **한 줄 한 줄 무슨 일이 일어나는지** 설명 주석이 달려 있습니다.
> ① **위에서 아래로** 순서대로 내려가며 → ② 각 셀의 **주석을 먼저 읽고** →
> ③ ▶(Shift+Enter)로 **바로 실행**하세요. ✅ 표시가 나오면 잘 된 것입니다.

> 💡 주석은 **쉬운 직관(비유) → 기술적 근거** 순서로 적혀 있어요. 텐서 모양(shape)이
> 바뀌는 곳마다 `(B, S, d_model)` 처럼 표기해 두었으니, 형태가 어떻게 흘러가는지 눈으로 따라가 보세요.
> (B=배치, S/T=시퀀스 길이, h=헤드 수, d_k=헤드당 차원)

> 🚀 **Colab에서 열기**: 이 `example-detailed.ipynb` 를 [Google Colab](https://colab.research.google.com) 에
> 업로드하거나, 저장소에 올린 뒤 `File ▸ Open notebook ▸ GitHub` 로 열면 됩니다.

트랜스포머의 핵심 직관을 한 문장으로 익힙니다 — 답 "**비**"를 만들려면 질문의 "**먹구름**"에 **주목**해야 합니다.
이게 바로 **어텐션**이 하는 일이에요. CPU로 수 초면 끝납니다(GPU 불필요).

## 0️⃣ 준비 — 라이브러리 & 한글 폰트

In [ ]:
# ── 0단계: 도구 챙기기 ──
# 요리로 치면 냄비·칼·도마를 꺼내는 단계입니다. 트랜스포머를 만들기 전에 필요한
# 라이브러리를 불러오고, 그래프에 한글이 깨지지 않도록 폰트를 준비합니다.

# 맨 앞의 '!'는 파이썬 코드가 아니라 '셸(터미널) 명령'을 실행하라는 Colab/Jupyter 문법입니다.
# 나눔고딕 폰트를 설치합니다(그래프 축에 한글을 표시하기 위함). Colab 전용이며,
# 로컬에서는 무시되거나 실패해도 아래 try/except 폴백이 대신 처리합니다.
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1   # 그래프용 한글 폰트 (Colab)
# math=sqrt·log 같은 수학 함수, random=무작위, logging=경고 메시지 제어에 사용합니다.
import math, random, logging                            # 기본 도구

# torch: 텐서(다차원 배열) 연산과 자동미분(backprop)을 제공하는 딥러닝 핵심 라이브러리.
import torch                                            # 딥러닝 프레임워크

# nn: Linear·LayerNorm·Embedding·Dropout 등 '신경망 부품'이 모여 있는 하위 모듈.
# - Linear: 선형 변환(행렬 곱) 레이어 - 입력벡터 * 가중치 + 편향(y = xW^T + b)
# - LayerNorm: 입력 벡터를 평균 0, 분산 1로 정규화하는 레이어
# - Embedding: 단어를 벡터로 바꾸는 레이어
# - Dropout: 학습 시 일부 뉴런을 무작위로 학습에서 제외하는 레이어
from torch import nn                                    # 신경망 부품

# pad_sequence: 길이가 제각각인 문장들을 가장 긴 것에 맞춰 <pad>로 채워 직사각형 텐서로 만들어 줍니다.
from torch.nn.utils.rnn import pad_sequence             # 길이 맞추기(패딩)

# 어텐션 히트맵 등 그림을 그리기 위한 시각화 도구.
import matplotlib.pyplot as plt                         # 그래프

# 시스템에 설치된 글꼴을 등록/선택할 때 사용하는 보조 모듈.
from matplotlib import font_manager                     # 폰트 관리

# matplotlib이 수식 폰트 관련 사소한 경고를 쏟아내지 않도록 로그 레벨을 ERROR로 올립니다(동작엔 영향 없음).
logging.getLogger("matplotlib.mathtext").setLevel(logging.ERROR)  # 사소한 폰트 경고 숨김

# 축 눈금의 음수 부호(−)가 네모(□)로 깨지는 현상을 방지합니다.
plt.rcParams["axes.unicode_minus"] = False                        # 마이너스 기호 깨짐 방지

# 방금 설치한 나눔고딕 폰트 파일을 matplotlib에 직접 등록해 기본 글꼴로 지정하려는 시도.
try:
    font_manager.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"                   # Colab: 나눔고딕
# 폰트 파일이 없는 로컬 환경이면 위에서 예외가 나므로, OS에 흔한 한글 폰트를 순서대로 탐색합니다.
except Exception:
    for _n in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:    # 로컬 폴백
        # 실제로 설치돼 있는 폰트 이름 집합에 후보가 들어 있으면 채택하고, 찾는 즉시 반복을 멈춥니다.
        if _n in {f.name for f in font_manager.fontManager.ttflist}:
            plt.rcParams["font.family"] = _n
            break

# 여기까지 에러 없이 왔다면 준비 완료. 설치된 PyTorch 버전도 함께 찍어 환경을 확인합니다.
print("준비 완료! PyTorch", torch.__version__)

## 1️⃣ 데이터 — 질문 → 답 6쌍

문장 틀은 **똑같이** 두고 **'대상' 단어만** 바꿉니다(먹구름/별/해/…). 그래야 답을 가르는
유일한 단서가 '대상'이 되어, 모델이 **반드시 그 단어에 주목**하게 됩니다.

In [ ]:
# ── 1단계: 아주 작은 '질문→답' 데이터 만들기 ──
# 모델이 배울 교재입니다. 문장 '틀'은 똑같이 두고 '대상' 단어(먹구름/별/해/…)만 바꿨어요.
# 이렇게 하면 답을 가르는 단서가 오직 그 한 단어뿐이라, 모델이 반드시 거기에 '주목'하게 됩니다.
# DATA: (질문 문자열, 답 문자열) 튜플 6개를 담은 리스트.
DATA = [
    ("하늘에 먹구름이 보이면 뭐가 생각나", "비가 올 것 같아"),   # 핵심 쌍: '먹구름' → '비'
    ("하늘에 별이 보이면 뭐가 생각나", "밤이 깊었나 봐"),       # '별' → '밤'
    ("하늘에 해가 보이면 뭐가 생각나", "아침이 밝았구나"),       # '해' → '아침'
    ("하늘에 무지개가 보이면 뭐가 생각나", "비가 그쳤나 봐"),     # '무지개' → '비 그침'
    ("하늘에 눈송이가 보이면 뭐가 생각나", "겨울이 왔구나"),      # '눈송이' → '겨울'
    ("하늘에 노을이 보이면 뭐가 생각나", "저녁이 되었네"),        # '노을' → '저녁'
]
# 잘 담겼는지 앞 3쌍만 눈으로 확인합니다(슬라이싱 [:3]).
for q, a in DATA[:3]:
    print(q, "→", a)

In [ ]:
# 체크포인트: 데이터가 6쌍 모두 잘 들어갔는지 확인.
assert len(DATA) == 6
print("✅ 데이터 OK")

## 2️⃣ 토큰화 & 단어장

컴퓨터는 글자를 못 읽으니 **단어를 번호로** 바꿉니다. 문장을 공백으로 자르고(토큰화),
단어↔번호 사전(Vocab)을 만듭니다. `<pad>/<sos>/<eos>/<unk>` 는 특수 토큰이에요.

> 📖 **상세 설명**: [어휘 사전(Vocab) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/vocab.html)

In [ ]:
# ── 2단계: 토큰화 & 단어장(Vocab) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/vocab.html
# 컴퓨터는 글자를 직접 못 읽으므로 '단어 → 번호'로 바꿔야 합니다. 그 사전이 Vocab입니다.

# 특수 토큰 4종: <pad>=길이 맞춤용 빈칸, <sos>=문장 시작 신호, <eos>=문장 끝 신호,
# <unk>=사전에 없는 낯선 단어. 리스트 맨 앞에 두어 항상 같은 번호(0,1,2,3)를 갖게 합니다.
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"   # 특수 토큰
SPECIALS = [PAD, SOS, EOS, UNK]

# 토큰화 = 문장을 '단어 단위'로 쪼개기. 여기선 가장 단순하게 공백으로만 자릅니다.
def tokenize(s):
    return s.strip().split()                    # 공백 단위로 자르기

# Vocab: 이 데이터에 등장하는 단어들로 '번호↔단어' 양방향 사전을 만드는 클래스.
class Vocab:
    def __init__(self, sentences):
        # 모든 문장의 모든 토큰을 모아 중복 제거(set) 후 정렬 → 실행마다 번호가 고정됩니다(재현성).
        toks = sorted({t for s in sentences for t in tokenize(s)})
        # itos = index→string. 특수토큰(0~3) 다음에 실제 단어들을 이어 붙입니다.
        self.itos = SPECIALS + toks             # 번호 → 단어
        # stoi = string→index. itos를 뒤집어 '단어로 번호를 빠르게 찾는' 딕셔너리.
        self.stoi = {t: i for i, t in enumerate(self.itos)}   # 단어 → 번호
    # len(vocab) 하면 전체 단어 수(임베딩 행 개수)가 나오도록 정의.
    def __len__(self):
        return len(self.itos)
    # 자주 쓰는 특수토큰 번호를 property로 편하게 꺼내 씁니다(예: vocab.pad_id).
    @property
    def pad_id(self):
        return self.stoi[PAD]
    @property
    def sos_id(self):
        return self.stoi[SOS]
    @property
    def eos_id(self):
        return self.stoi[EOS]
    # encode: 문장 → 번호 리스트. 모르는 단어는 <unk> 번호로 대체(.get의 기본값 활용).
    def encode(self, s, add_special=True):
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in tokenize(s)]
        # add_special=True면 앞뒤에 <sos>…<eos>를 붙여 '문장 시작/끝'을 모델에 알려 줍니다.
        return [self.sos_id] + ids + [self.eos_id] if add_special else ids
    # decode: 번호 리스트 → 사람이 읽는 문장. 특수토큰은 눈에 안 보이게 걸러서 이어 붙입니다.
    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if self.itos[i] not in (PAD, SOS, EOS))

In [ ]:
# 질문(source)과 답(target)은 쓰는 단어가 다르므로 단어장을 각각 따로 만듭니다.
# src_vocab: 질문 쪽 단어장. DATA의 각 튜플에서 질문 q만 뽑아(리스트 컴프리헨션) 넘깁니다.
src_vocab = Vocab([q for q, _ in DATA])         # 질문 단어장
# tgt_vocab: 답 쪽 단어장. 답 a만 뽑아 만듭니다.
tgt_vocab = Vocab([a for _, a in DATA])         # 답 단어장
# 각 단어장 크기 = 특수토큰 4개 + 고유 단어 수. 이 값이 임베딩/출력층 크기를 결정합니다.
print("질문 단어", len(src_vocab), "| 답 단어", len(tgt_vocab))

In [ ]:
# 체크포인트: 특수토큰 번호가 기대대로인지 확인(<pad>=0, <sos>=1). 사전 구성이 올바른지 검증.
assert src_vocab.pad_id == 0 and tgt_vocab.sos_id == 1
print("✅ 사전 OK")

## 3️⃣ 모델 부품 만들기

이제 트랜스포머의 부품을 하나씩 직접 만듭니다. **위치 인코딩 → 어텐션 → 멀티헤드 →
FFN → 인코더/디코더 블록 → 마스크 → 전체 조립** 순서예요. 조금 길지만, 손으로 치면 구조가 몸에 남습니다. 💪

> 📖 **상세 설명**: [위치 인코딩(Positional Encoding) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/positional-encoding.html)

In [ ]:
# ── 3단계: 부품 ① 위치 인코딩(Positional Encoding) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/positional-encoding.html
# 어텐션은 단어들을 '동시에·순서 없이' 봅니다(순열 불변). 그래서 '몇 번째 단어인지'를
# 따로 알려 줘야 해요. sin/cos '파도무늬'를 임베딩에 더해 위치 정보를 심어 줍니다.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128, dropout=0.1):
        super().__init__()
        # 학습 안정화를 위한 드롭아웃(위치정보를 더한 뒤 일부를 무작위로 0으로).
        self.dropout = nn.Dropout(dropout)
        # pe: (max_len, d_model). 위치마다 d_model 길이의 고유한 '파도무늬' 벡터가 들어갈 빈 표.
        pe = torch.zeros(max_len, d_model)                      # 빈 표
        # pos: (max_len, 1). 위치 인덱스 0,1,2,…를 세로로 세운 것. 뒤의 div와 브로드캐스팅됩니다.
        pos = torch.arange(0, max_len).float().unsqueeze(1)     # 위치 0,1,2,...
        # div: (d_model/2,). 차원마다 다른 '주파수'. 낮은 차원=느린 파도, 높은 차원=빠른 파도.
        # exp(−log(10000)·(2i/d_model)) = 10000^(−2i/d_model) 를 수치적으로 안정하게 계산한 형태.
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # 짝수 차원(0,2,4,…)에는 sin, 홀수 차원(1,3,5,…)에는 cos를 채웁니다.
        # pos*div의 브로드캐스팅: (max_len,1)*(d_model/2,) → (max_len, d_model/2).
        pe[:, 0::2] = torch.sin(pos * div)     # 짝수 차원 = sin
        pe[:, 1::2] = torch.cos(pos * div)     # 홀수 차원 = cos
        # register_buffer: 학습되지 않는 상수 텐서를 모듈에 등록(=nn.Parameter와 달리 기울기 학습 X).
        # 대신 .to(device)·state_dict 저장 시 파라미터처럼 함께 따라다닙니다. (1, max_len, d_model)로 배치축 추가.
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        # x: (B, S, d_model) 임베딩. 앞에서 만든 위치표에서 현재 문장 길이 S만큼 잘라 더합니다.
        # pe[:, :S]는 (1, S, d_model) → 배치 B 전체에 브로드캐스팅되어 같은 위치정보가 더해집니다.
        return self.dropout(x + self.pe[:, :x.size(1)])   # 임베딩 + 위치 파도

> 📖 **상세 설명**: [어텐션(Scaled Dot-Product Attention) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/attention.html)

In [ ]:
# ── 3단계: 부품 ② 어텐션 핵심 공식 (Scaled Dot-Product Attention) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/attention.html
# 직관: 각 단어(Query)가 다른 단어들(Key)과 얼마나 '관련 있나'를 점수로 매기고,
# 그 점수를 비율(softmax)로 바꿔 값(Value)들을 '가중 평균'합니다. 곧 중요한 곳에 밑줄 긋기.
# q,k,v shape: (B, h, S, d_k). mask는 가릴 위치를 알려 주는 불리언 텐서(옵션).
def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    # d_k = 헤드 하나의 차원 수. 스케일링에 쓰입니다.
    d_k = q.size(-1)
    # 관련도 점수 = Q·Kᵀ. k.transpose(-2,-1): (B,h,S_k,d_k)→(B,h,d_k,S_k). 곱 결과 (B,h,S_q,S_k).
    # √d_k로 나누는 이유: 차원이 크면 내적 값이 커져 softmax가 한쪽으로 포화(거의 0/1)되고,
    # 그러면 기울기가 사라져(gradient vanishing) 학습이 안 됩니다. √d_k로 스케일을 되돌려 줍니다.
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)   # 관련도 점수
    if mask is not None:
        # 가려야 할 자리(mask==0)의 점수를 −무한대로 만듭니다. 다음 줄 softmax에서 exp(−inf)=0이 되어,
        # 결국 그 위치의 가중치가 정확히 0 → '보지 않음'이 구현됩니다(패딩/미래 토큰 차단).
        scores = scores.masked_fill(mask == 0, float("-inf"))        # 가릴 곳은 -무한대
    # dim=-1(마지막 축=Key 방향)으로 softmax → 각 Query가 모든 Key에 준 가중치의 합이 1이 됩니다.
    attn = torch.softmax(scores, dim=-1)                             # 합=1 비율로
    if dropout is not None:
        # 어텐션 가중치 일부를 무작위로 0으로 떨궈 특정 연결에 과의존하지 않게 합니다(정규화).
        attn = dropout(attn)
    # 가중치(attn)로 V를 가중합: (B,h,S_q,S_k)·(B,h,S_k,d_k)→(B,h,S_q,d_k). attn은 시각화용으로 함께 반환.
    return torch.matmul(attn, v), attn                               # V를 비율대로 가중합

In [ ]:
# 체크포인트: 어텐션 공식 검증. Q·K 점수가 모두 같으면(여기선 전부 1) 두 Key에 0.5씩 균등 분배돼야 함.
_c, _w = scaled_dot_product_attention(torch.ones(1, 1, 2, 4), torch.ones(1, 1, 2, 4),
                                      torch.arange(8.).reshape(1, 1, 2, 4))
assert torch.allclose(_w, torch.full((1, 1, 2, 2), 0.5))   # 점수가 같으면 0.5씩 균등
print("✅ 어텐션 공식 OK")

> 📖 **상세 설명**: [멀티헤드 어텐션(Multi-Head Attention) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/multi-head-attention.html)

In [ ]:
# ── 3단계: 부품 ③ 멀티헤드 어텐션 (Multi-Head Attention) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/multi-head-attention.html
# 직관: 한 번에 한 관점만 보지 말고, 여러 '헤드'로 나눠 서로 다른 관점(문법/의미/위치 등)에서
# 동시에 주목한 뒤 합칩니다. 카메라 여러 대로 같은 장면을 다른 각도에서 찍는 셈.
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        # h=헤드 수, dk=헤드당 차원. d_model을 h등분하므로 d_model = h * dk가 되어야 합니다.
        self.h, self.dk = num_heads, d_model // num_heads
        # Q·K·V를 각각 만드는 선형변환. 입출력 모두 d_model이라 헤드로 쪼개도 총 차원은 그대로.
        self.wq = nn.Linear(d_model, d_model); self.wk = nn.Linear(d_model, d_model)
        # wv=Value 변환, wo=여러 헤드 결과를 합친 뒤 섞어 주는 출력 변환.
        self.wv = nn.Linear(d_model, d_model); self.wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        # 학습엔 안 쓰고, 나중에 히트맵을 그리려고 마지막 어텐션 가중치를 잠깐 보관해 둘 자리.
        self.last_attn_weights = None                 # 시각화용 보관
    # split: (B, S, d_model) → (B, h, S, dk). 헤드축을 앞으로 빼서 헤드별로 따로 어텐션하게 만듭니다.
    def split(self, x):
        b, s, _ = x.shape
        # view로 마지막 차원을 (h, dk)로 쪼갠 뒤 transpose로 (배치, 헤드, 길이, dk) 순서로 재배열.
        return x.view(b, s, self.h, self.dk).transpose(1, 2)   # (B, 헤드, 길이, dk)
    # merge: split의 역연산. (B, h, S, dk) → (B, S, d_model)으로 헤드들을 다시 이어 붙입니다.
    def merge(self, x):
        b, _, s, _ = x.shape
        # transpose로 (B, S, h, dk) 순서로 되돌린 뒤, view로 마지막 두 축을 h*dk=d_model로 합칩니다.
        # contiguous(): transpose는 메모리를 실제로 옮기지 않고 '보는 순서'만 바꾸므로, view가 요구하는
        # 연속 메모리 조건을 맞추려면 여기서 실제 복사가 필요합니다.
        return x.transpose(1, 2).contiguous().view(b, s, self.h * self.dk)
    def forward(self, q, k, v, mask=None):
        # 입력을 각각 선형변환한 뒤 헤드로 쪼갭니다. 결과 shape: (B, h, S, dk).
        q, k, v = self.split(self.wq(q)), self.split(self.wk(k)), self.split(self.wv(v))
        # 헤드별로 앞서 만든 어텐션 공식을 적용. ctx=(B,h,S,dk), attn=(B,h,S_q,S_k).
        ctx, attn = scaled_dot_product_attention(q, k, v, mask, self.dropout)
        # detach(): 시각화용 복사본은 계산그래프에서 분리해 메모리 낭비/역전파 오염을 막습니다.
        self.last_attn_weights = attn.detach()
        # 헤드들을 merge로 합친 뒤 wo로 한 번 더 섞어 최종 출력 (B, S, d_model)을 냅니다.
        return self.wo(self.merge(ctx))

In [ ]:
# 체크포인트: 멀티헤드가 입력 shape (1,3,8)을 그대로 (1,3,8)로 돌려주는지(차원 보존) 확인.
_m = MultiHeadAttention(8, 4, 0.0)
assert _m(torch.randn(1, 3, 8), torch.randn(1, 3, 8), torch.randn(1, 3, 8)).shape == (1, 3, 8)
print("✅ 멀티헤드 OK")

> 📖 **상세 설명**: [FFN(Position-wise Feed-Forward) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/feed-forward.html)

In [ ]:
# ── 3단계: 부품 ④ FFN (Position-wise Feed-Forward) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/feed-forward.html
# 직관: 어텐션이 '단어들끼리 정보를 섞는' 단계라면, FFN은 각 단어가 '혼자 더 깊이 곱씹는' 단계.
# 모든 위치(단어)에 '똑같은' 작은 2층 신경망을 독립적으로 적용합니다(그래서 position-wise).
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # Linear(d_model→d_ff)로 차원을 크게 늘려(보통 4배) 표현력을 키우고 → ReLU로 비선형성 부여 →
        # Dropout으로 과적합 억제 → Linear(d_ff→d_model)로 원래 차원으로 되돌립니다.
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x):
        # x: (B, S, d_model) → (B, S, d_model). 각 위치가 서로 독립적으로 같은 net을 통과합니다.
        return self.net(x)

> 📖 **상세 설명**: [인코더 블록(EncoderLayer) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/encoder-layer.html)

In [ ]:
# ── 3단계: 부품 ⑤ 인코더 블록 (EncoderLayer) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/encoder-layer.html
# 구조: 셀프어텐션 → (원본 더하기+정규화) → FFN → (더하기+정규화). 질문을 '이해'하는 한 겹.
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # 셀프어텐션: 문장 안의 단어들끼리 서로 참조합니다.
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        # LayerNorm 2개: 각 하위층 뒤에서 출력 분포를 정규화해 깊은 망의 학습을 안정화합니다.
        self.n1 = nn.LayerNorm(d_model); self.n2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, mask):
        # 잔차 연결(x + …): 원본 입력을 그대로 더해 주어, 층이 깊어져도 기울기가 잘 흐르고
        # '아무것도 안 바꾸기'가 쉬워 학습이 안정됩니다. 그 합을 LayerNorm으로 정규화(Post-LN 방식).
        # 셀프어텐션이라 q=k=v=x. mask는 패딩 자리를 가립니다.
        x = self.n1(x + self.drop(self.attn(x, x, x, mask)))   # 잔차 + 정규화
        # 두 번째 하위층: FFN에도 동일한 (잔차 + 정규화) 패턴을 적용.
        x = self.n2(x + self.drop(self.ffn(x)))
        return x

> 📖 **상세 설명**: [디코더 블록(DecoderLayer) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/decoder-block.html)

In [ ]:
# ── 3단계: 부품 ⑥ 디코더 블록 (DecoderLayer) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/decoder-block.html
# 구조: 마스크드 셀프어텐션 → 크로스어텐션(질문 곁눈질) → FFN. 답을 '생성'하는 한 겹.
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # self_attn: 지금까지 만든 답 단어들끼리 참조(단, 미래는 못 보게 마스킹).
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        # cross_attn: Query=답, Key/Value=인코더가 이해한 질문. 답을 쓰며 질문을 '곁눈질'하는 부분.
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        # 하위층이 3개라 LayerNorm도 3개.
        self.n1 = nn.LayerNorm(d_model); self.n2 = nn.LayerNorm(d_model); self.n3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, enc, tgt_mask, src_mask):
        # ① 마스크드 셀프어텐션: q=k=v=x(생성 중인 답). tgt_mask가 '미래 단어+패딩'을 가려
        #    답을 만들 때 정답을 미리 훔쳐보지 못하게 합니다(정답 누설 방지).
        x = self.n1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))       # 미래 가림
        # ② 크로스어텐션: q=x(답), k=v=enc(질문 인코딩). src_mask는 질문의 패딩 자리를 가립니다.
        x = self.n2(x + self.drop(self.cross_attn(x, enc, enc, src_mask)))  # 질문 곁눈질
        # ③ FFN: 각 위치를 혼자 곱씹기. 세 하위층 모두 (잔차 + 정규화) 패턴 동일.
        x = self.n3(x + self.drop(self.ffn(x)))
        return x

> 📖 **상세 설명**: [인코더(Encoder) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/encoder.html)

In [ ]:
# ── 3단계: 부품 ⑦ 인코더 (Encoder) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/encoder.html
# 임베딩 + 위치인코딩으로 질문을 벡터열로 바꾼 뒤, 인코더 블록을 여러 겹 통과시켜 질문을 이해합니다.
class Encoder(nn.Module):
    def __init__(self, vocab, d_model, num_heads, num_layers, d_ff, dropout, max_len):
        super().__init__()
        # sqrt(d_model) 스케일링에서 쓰려고 d_model을 보관.
        self.d_model = d_model
        # 단어 번호 → d_model 차원 벡터로 바꾸는 임베딩 표. shape (vocab, d_model).
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        # 같은 구조의 인코더 블록을 num_layers개 쌓습니다. ModuleList로 담아야 파라미터가 제대로 등록됩니다.
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout)
                                     for _ in range(num_layers)])
    def forward(self, src, mask):
        # 임베딩에 √d_model을 곱하는 이유: 이어서 더하는 위치인코딩(진폭 ~1)과 크기의 균형을 맞춰,
        # 임베딩 신호가 위치정보에 묻히지 않게 합니다(원 논문 관례). 그 뒤 위치인코딩을 더합니다.
        x = self.pos(self.emb(src) * math.sqrt(self.d_model))
        # 블록들을 순서대로 통과. 각 블록이 문맥 정보를 조금씩 더 녹여 냅니다.
        for l in self.layers:
            x = l(x, mask)
        return x

> 📖 **상세 설명**: [디코더(Decoder) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/decoder.html)

In [ ]:
# ── 3단계: 부품 ⑧ 디코더 (Decoder) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/decoder.html
# 인코더와 뼈대는 같지만, 각 층에서 인코딩된 질문(enc)을 참고해 '답을 만들 표현'으로 바꿉니다.
class Decoder(nn.Module):
    def __init__(self, vocab, d_model, num_heads, num_layers, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model
        # 답 단어장 기준 임베딩. 나중에 이 표의 가중치를 출력층과 공유(weight tying)합니다.
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        # 디코더 블록을 num_layers개 쌓기.
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout)
                                     for _ in range(num_layers)])
    def forward(self, tgt, enc, tgt_mask, src_mask):
        # 인코더와 동일하게 임베딩×√d_model + 위치인코딩으로 시작.
        x = self.pos(self.emb(tgt) * math.sqrt(self.d_model))
        # 각 디코더 블록에 '인코딩된 질문(enc)'과 두 종류의 마스크를 함께 넘깁니다.
        for l in self.layers:
            x = l(x, enc, tgt_mask, src_mask)
        return x

> 📖 **상세 설명**: [Transformer(전체 조립) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/transformer-assembly.html)

In [ ]:
# ── 3단계: 부품 ⑨ 전체 조립 (Transformer) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/transformer-assembly.html
# 인코더 + 디코더 + 출력층을 하나로 묶습니다. 하이퍼파라미터 기본값도 여기서 정합니다.
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, num_heads=4,
                 num_layers=2, d_ff=256, dropout=0.1, max_len=32):
        super().__init__()
        # 질문을 이해하는 인코더.
        self.encoder = Encoder(src_vocab, d_model, num_heads, num_layers, d_ff, dropout, max_len)
        # 답을 생성하는 디코더.
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, num_layers, d_ff, dropout, max_len)
        # 출력층: d_model 벡터를 답 단어장 크기의 점수(logit)로 바꿉니다. bias=False는 아래 가중치공유 때문.
        self.out = nn.Linear(d_model, tgt_vocab, bias=False)
        # weight tying: 출력층 가중치를 디코더 임베딩 표와 '같은 텐서'로 공유합니다.
        # 입력 임베딩과 출력 예측이 같은 단어공간을 쓰게 해 파라미터를 줄이고 일반화를 돕습니다.
        self.out.weight = self.decoder.emb.weight     # weight tying(가중치 공유)
    def forward(self, src, tgt, src_mask, tgt_mask):
        # 학습용 전체 경로: 질문 인코딩 → 디코더 통과 → 출력층으로 각 위치의 다음 단어 점수 산출.
        return self.out(self.decoder(tgt, self.encoder(src, src_mask), tgt_mask, src_mask))
    # 추론 때는 질문 인코딩을 한 번만 하면 되므로 따로 떼어 둔 함수.
    def encode(self, src, src_mask):
        return self.encoder(src, src_mask)
    # 추론 때 답을 한 단어씩 늘려 가며 매 스텝 호출하는 함수(인코딩 결과 enc는 재사용).
    def decode_step(self, tgt, enc, tgt_mask, src_mask):
        return self.out(self.decoder(tgt, enc, tgt_mask, src_mask))

> 📖 **상세 설명**: [마스킹(Masking) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/masking.html)

In [ ]:
# ── 3단계: 부품 ⑩ 마스크 만들기 ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/masking.html
# 마스크 = '여기는 보지 마'라고 알려 주는 True/False 표. 두 종류를 씁니다.
# make_padding_mask: 길이 맞추려고 넣은 <pad> 자리를 가립니다. seq shape (B, S).
def make_padding_mask(seq, pad_id):
    # (seq != pad_id): 진짜 단어=True, 패딩=False인 (B,S) 불리언. unsqueeze 두 번으로 (B,1,1,S).
    # 어텐션 점수 (B,h,S_q,S_k)에 브로드캐스팅되어 'Key가 패딩인 열'을 전부 가립니다.
    return (seq != pad_id).unsqueeze(1).unsqueeze(2)              # 패딩 위치 가림
# make_causal_mask: 미래 단어를 못 보게 하는 삼각형 마스크(causal=인과). 정답 누설을 막습니다.
def make_causal_mask(seq_len, device):
    # torch.tril = 하삼각(대각선 포함) 1, 나머지 0 → i번째 단어는 0..i까지만 볼 수 있음.
    # .bool()로 True/False화, unsqueeze 두 번으로 (1,1,L,L) 만들어 배치·헤드축에 브로드캐스팅.
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).bool().unsqueeze(0).unsqueeze(0)
# make_decoder_mask: 디코더 셀프어텐션엔 두 마스크가 모두 필요합니다.
def make_decoder_mask(tgt, pad_id):
    # &(논리곱): 패딩도 아니고 미래도 아닌 자리만 True. (B,1,1,T) & (1,1,T,T) → (B,1,T,T)로 브로드캐스팅.
    return make_padding_mask(tgt, pad_id) & make_causal_mask(tgt.size(1), tgt.device)

In [ ]:
# 체크포인트: 3x3 인과 마스크의 True 개수가 1+2+3=6인지 확인(하삼각형이 제대로 만들어졌는지).
assert make_causal_mask(3, torch.device("cpu")).sum().item() == 6   # 1+2+3
print("✅ 마스크 OK")

## 4️⃣ 학습 — "틀린 만큼 조금씩 고치기"

학습은 **(1) 예측 → (2) 정답과 비교해 loss 계산 → (3) loss만큼 가중치 수정** 의 반복입니다.
loss 숫자가 점점 0에 가까워지면 잘 배우는 중이에요.

In [ ]:
# ── 4단계: 배치 만들기 (build_batches) ──
# 질문/답 문장들을 번호 텐서로 바꾸고, 길이를 맞춰 하나의 직사각형 배치로 묶습니다.
def build_batches(pairs, sv, tv):
    # 질문(src): 특수토큰 없이 인코딩. 각 문장을 텐서로 만든 리스트(길이 제각각).
    src = [torch.tensor(sv.encode(q, add_special=False)) for q, _ in pairs]
    # 답(tgt): <sos>…<eos>를 붙여 인코딩(생성 시작/끝 신호가 필요하므로).
    tgt = [torch.tensor(tv.encode(a, add_special=True)) for _, a in pairs]
    # pad_sequence: 가장 긴 문장에 맞춰 나머지를 pad_id로 채워 (B, 최대길이) 텐서로. batch_first=배치축이 앞.
    src = pad_sequence(src, batch_first=True, padding_value=sv.pad_id)   # 길이 맞추기
    tgt = pad_sequence(tgt, batch_first=True, padding_value=tv.pad_id)
    return src, tgt

> 📖 **상세 설명**: [학습(Training Loop) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/training-loop.html)

In [ ]:
# ── 4단계: 모델 만들고 학습하기 ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/training-loop.html
# 학습 = (1) 예측 → (2) 정답과 비교해 loss 계산 → (3) loss만큼 가중치 수정, 을 반복하는 것.
# 같은 결과가 나오도록 난수 시드를 고정합니다(가중치 초기화·드롭아웃이 매번 동일해짐).
torch.manual_seed(42)                                    # 재현성(같은 결과)
# 앞서 만든 함수로 질문/답 배치 텐서를 준비.
src_ids, tgt_ids = build_batches(DATA, src_vocab, tgt_vocab)
# 위치인코딩 표가 넉넉하도록 max_len을 데이터 최대 길이+여유로 잡습니다.
max_len = max(src_ids.size(1), tgt_ids.size(1), 20) + 2
# 단어장 크기를 넘겨 모델 생성(임베딩·출력층 크기가 여기서 정해짐).
model = Transformer(len(src_vocab), len(tgt_vocab), max_len=max_len)
# Adam: 가중치를 자동으로 갱신해 주는 옵티마이저. lr=학습률(한 걸음 크기).
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
# 손실함수: 예측 분포와 정답 단어의 차이를 측정. ignore_index로 <pad> 자리는 손실 계산에서 제외.
crit = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_id)
# 티처 포싱: 디코더 입력은 <sos>부터(마지막 뺀 앞부분), 정답은 한 칸 뒤(첫 칸 뺀 뒷부분).
# 즉 "지금까지의 답"을 보고 "바로 다음 단어"를 맞히도록 한 칸 어긋나게 짝지어 줍니다.
dec_in, dec_tgt = tgt_ids[:, :-1], tgt_ids[:, 1:]        # 입력은 <sos>부터, 정답은 한 칸 뒤
# 질문 쪽 패딩 마스크는 매 스텝 동일하므로 루프 밖에서 한 번만 계산.
src_mask = make_padding_mask(src_ids, src_vocab.pad_id)
# 학습 모드로 전환(드롭아웃·정규화가 학습용으로 동작).
model.train()
# 400번 반복 학습. 데이터가 아주 작아 CPU로도 수 초면 끝납니다.
for ep in range(1, 401):
    # 디코더 마스크는 미래 가림+패딩 가림을 함께. dec_in 길이에 맞춰 매번 생성.
    tgt_mask = make_decoder_mask(dec_in, tgt_vocab.pad_id)
    # (1) 예측: 각 위치의 다음 단어 점수(logits) (B, T, 답단어수).
    logits = model(src_ids, dec_in, src_mask, tgt_mask)                    # (1) 예측
    # (2) 비교: (B*T, 단어수)와 (B*T,)로 펼쳐 CrossEntropy 계산. reshape는 배치·시간축을 한 줄로 합치는 것.
    loss = crit(logits.reshape(-1, logits.size(-1)), dec_tgt.reshape(-1))  # (2) 정답과 비교
    # (3) 수정: 기울기 초기화 → 역전파로 기울기 계산 → 옵티마이저가 가중치를 한 걸음 갱신.
    opt.zero_grad(); loss.backward(); opt.step()                          # (3) 가중치 수정
    # 100번마다(그리고 첫 에폭) loss를 출력해 잘 내려가는지 확인.
    if ep % 100 == 0 or ep == 1:
        print(f"epoch {ep:4d} | loss {loss.item():.4f}")

In [ ]:
# 체크포인트: 학습이 충분히 됐는지 최종 loss가 0.5 미만인지 확인.
assert loss.item() < 0.5
print("✅ 학습 완료! 최종 loss", round(loss.item(), 4))

## 5️⃣ 답 만들기 — 한 단어씩(마스킹)

답은 `<sos>`부터 **한 단어씩** 만듭니다. 다음 단어를 고를 땐 **앞말만** 볼 수 있어요(미래는 마스킹).
끝말잇기랑 비슷하죠!

> 📖 **상세 설명**: [답 생성(자기회귀 디코딩) 설명 보기](https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/greedy-decoding.html)

In [ ]:
# ── 5단계: 답 생성 함수 (answer) ──
# 📖 상세 설명: https://htmlpreview.github.io/?https://github.com/unicorn-campus/mini-transformer/blob/main/explain/greedy-decoding.html
# 추론은 답을 '한 단어씩' 만들어 갑니다(자기회귀). 매 스텝 가장 점수 높은 단어를 고르는 greedy 방식.
# @torch.no_grad(): 추론 중엔 기울기 계산이 필요 없으니 꺼서 메모리·속도를 아낍니다.
@torch.no_grad()
def answer(model, sv, tv, question, max_len=20):
    # 평가 모드(드롭아웃 끔 → 결과가 결정적).
    model.eval()
    # 질문을 번호 텐서로. [ ]로 감싸 배치축을 만들어 (1, S_src)로.
    src = torch.tensor([sv.encode(question, add_special=False)])
    src_mask = make_padding_mask(src, sv.pad_id)
    # 질문 인코딩은 한 번만 하고 이후 스텝에서 재사용.
    enc = model.encode(src, src_mask)
    # gen: 지금까지 생성한 답 번호들(<sos>로 시작). steps: 시각화를 위한 (지금까지답, 다음단어) 기록.
    gen, steps = [tv.sos_id], []
    for _ in range(max_len):
        # 지금까지의 답을 (1, 현재길이) 텐서로.
        tgt = torch.tensor([gen])
        # 추론 땐 패딩이 없으니 인과(미래 가림) 마스크만 있으면 됩니다.
        tgt_mask = make_causal_mask(tgt.size(1), tgt.device)
        # 다음 단어 점수 계산 (1, 현재길이, 답단어수).
        logits = model.decode_step(tgt, enc, tgt_mask, src_mask)
        # 마지막 위치(-1)의 점수 중 최고점 단어를 선택(greedy). .item()으로 파이썬 int로.
        nxt = logits[0, -1].argmax().item()               # 마지막 위치의 최고점 단어
        # 이번 스텝의 (지금까지 만든 답, 새로 고른 단어)를 기록.
        steps.append((tv.decode(gen), tv.itos[nxt]))
        # 고른 단어를 답 뒤에 이어 붙입니다(다음 스텝 입력이 됨).
        gen.append(nxt)
        # <eos>가 나오면 문장이 끝난 것이므로 생성 중단.
        if nxt == tv.eos_id:
            break
    # 완성된 답 전체를 한 번 더 디코더에 통과시켜, 마지막 스텝 기준의 어텐션 가중치를 확보.
    tgt = torch.tensor([gen]); tgt_mask = make_causal_mask(tgt.size(1), tgt.device)
    model.decode_step(tgt, enc, tgt_mask, src_mask)       # 어텐션 확보용 한 번 더
    # 마지막 디코더 층의 크로스어텐션 가중치를 꺼내 헤드 평균(mean(0))을 냅니다.
    # [0]=배치 첫 문장. 결과 shape (답길이, 질문길이) → 히트맵의 재료.
    cross = model.decoder.layers[-1].cross_attn.last_attn_weights[0].mean(0)
    # (스텝기록, 완성된 답 문자열, 크로스어텐션)을 반환.
    return steps, tv.decode(gen), cross

In [ ]:
# 먹구름 질문을 넣어 봅니다.
q = "하늘에 먹구름이 보이면 뭐가 생각나"
# answer가 (한 단어씩 만든 과정, 최종 답, 어텐션)을 돌려줍니다.
steps, ans, cross = answer(model, src_vocab, tgt_vocab, q)
print("질문:", q)
# 각 스텝에서 '지금까지의 답 → 새로 고른 단어'를 출력해 생성 과정을 눈으로 봅니다.
for seen, nxt in steps:
    print(f"  '{seen or '<sos>'}'  →  {nxt}")
print("답:", ans)

## 6️⃣ 어텐션 히트맵 — 답이 어디에 주목했나?

이제 핵심! 답의 각 단어가 **질문의 어느 단어를 봤는지** 색으로 봅니다.
'먹구름이' 열이 가장 **밝으면** 성공 — 모델이 먹구름에 주목해 '비'를 떠올린 거예요. 🌧️

In [ ]:
# ── 6단계: 어텐션 히트맵 함수 (show_attention) ──
# 답의 각 단어가 질문의 어느 단어에 주목했는지 색으로 보여 줍니다(밝을수록 크게 주목).
def show_attention(question, answer_text, cross):
    # 축 눈금에 쓸 질문/답 토큰 리스트.
    qs, ans_toks = tokenize(question), tokenize(answer_text)
    # cross(답길이×질문길이)에서 실제 토큰 개수만큼 잘라 numpy 배열로. (특수토큰 여백 제거)
    w = cross[:len(ans_toks), :len(qs)].numpy()
    # 토큰 수에 비례해 그림 크기를 잡아 글자가 겹치지 않게 합니다.
    plt.figure(figsize=(1.1 * len(qs) + 1, 0.7 * len(ans_toks) + 1))
    # 가중치 행렬을 색 그리드로. viridis=낮음(보라)~높음(노랑) 컬러맵.
    plt.imshow(w, aspect="auto", cmap="viridis")
    # x축=질문 토큰(라벨 겹침 방지로 30도 회전), y축=생성한 답 토큰.
    plt.xticks(range(len(qs)), qs, rotation=30, ha="right")
    plt.yticks(range(len(ans_toks)), ans_toks)
    plt.xlabel("질문 토큰"); plt.ylabel("생성한 답 토큰")
    plt.title(f"'{answer_text}' 이(가) 주목한 곳")
    # 색↔값 대응 막대, 여백 정리 후 화면에 출력.
    plt.colorbar(); plt.tight_layout(); plt.show()

In [ ]:
# 앞에서 얻은 q(질문), ans(답), cross(어텐션)로 히트맵을 그립니다.
# '먹구름이' 열이 가장 환하면 모델이 그 단어에 주목해 '비'를 떠올린 것!
show_attention(q, ans, cross)

## 7️⃣ 대조 — 키워드가 바뀌면 답도 바뀐다

문장 틀은 똑같은데 '대상' 단어만 바꾸면, 모델이 주목하는 곳이 옮겨가고 답도 달라집니다.

In [ ]:
# ── 7단계: 키워드를 바꿔 답 비교하기 ──
# 문장 틀은 그대로 두고 '대상' 단어만 바꾸면, 주목하는 곳이 옮겨가고 답도 달라집니다.
for q in ["하늘에 먹구름이 보이면 뭐가 생각나",
          "하늘에 별이 보이면 뭐가 생각나",
          "하늘에 해가 보이면 뭐가 생각나"]:
    # 답만 필요하므로 steps·cross는 _로 버립니다.
    _, a, _ = answer(model, src_vocab, tgt_vocab, q)
    # tokenize(q)[1] = 질문의 두 번째 토큰(=바뀌는 '대상' 단어). :<6은 왼쪽 정렬 6칸 폭.
    print(f"{tokenize(q)[1]:<6} → {a}")

## 8️⃣ 정리 & 다음 단계 🎉

수고했어요! 방금 여러분은 트랜스포머를 **손으로 직접 만들어** 봤습니다.

- **어텐션** = 답을 만들 때 중요한 단어(먹구름)에 **밑줄 긋기**
- **마스킹 디코딩** = 답을 **한 단어씩**, 미래는 못 보고 만들기
- **크로스 어텐션** = 답을 쓰는 내내 **질문을 곁눈질**

➡️ 다음은 본 과제 **한국어→영어 번역 미니 트랜스포머**입니다. 구조는 똑같고,
**질문→답**이 **한국어→영어**로 바뀔 뿐이에요. 여기서 익힌 감각으로 바로 도전해 보세요!